In [1]:
import zipfile, os, re
from pathlib import Path
import pandas as pd
import numpy as np

# ----------------------------
# CONFIG: paths to your zip files
# ----------------------------
zip_paths = [
    "Assessments.zip",
    "Enrollment.zip",
    "Subject_Characteristics.zip",
    "Assessments (1).zip"
]
extract_root = "/mnt/data/extracted_all"
merged_out = "/mnt/data/adni_merged_features.csv"

# ----------------------------
# STEP 1: unzip everything
# ----------------------------
os.makedirs(extract_root, exist_ok=True)
for zp in zip_paths:
    if not os.path.exists(zp): 
        continue
    with zipfile.ZipFile(zp, "r") as z:
        z.extractall(extract_root)

# ----------------------------
# STEP 2: helper to find CSV by regex
# ----------------------------
def find_csv(pattern):
    rx = re.compile(pattern, re.I)
    for root, _, files in os.walk(extract_root):
        for f in files:
            if rx.search(f) and f.endswith(".csv"):
                return os.path.join(root, f)
    return None

# ----------------------------
# STEP 3: load key tables
# ----------------------------
tables = {}
targets = {
    "DXSUM": r"DXSUM",
    "ADAS": r"ADAS_\d",
    "MMSE": r"MMSE_\d",
    "MOCA": r"MOCA_\d",
    "CDR": r"CDR_\d",
    "FAQ": r"FAQ_\d",
    "ECOGPT": r"ECOGPT_\d",
    "ECOGSP": r"ECOGSP_\d",
    "GDSCALE": r"GDSCALE_\d",
    "NPIQ": r"NPIQ_\d",
    "CBBRESULTS": r"CBBRESULTS_\d",
    "PTDEMOG": r"PTDEMOG_\d"
}

for k, pat in targets.items():
    p = find_csv(pat)
    if p:
        try:
            df = pd.read_csv(p, low_memory=False)
            tables[k] = df
            print(f"Loaded {k}: {df.shape} from {p}")
        except Exception as e:
            print(f"Error reading {p}: {e}")

# ----------------------------
# STEP 4: normalize keys (RID + VISCODE2)
# ----------------------------
def prep(df):
    df = df.copy()
    # find RID
    rid_col = [c for c in df.columns if c.upper()=="RID"]
    if rid_col:
        rid_col = rid_col[0]
    elif "PTID" in df.columns:
        df["RID"] = df["PTID"].astype(str).str.extract(r"(\d+)$")
        rid_col = "RID"
    else:
        return None
    # find VISCODE2 or VISCODE
    vcol = None
    for c in df.columns:
        if "VISCODE2" in c.upper():
            vcol = c; break
    if not vcol:
        for c in df.columns:
            if c.upper()=="VISCODE":
                vcol = c; break
    if not vcol: return None
    df["visit_id"] = df[rid_col].astype(str) + "_" + df[vcol].astype(str)
    return df

for k in tables:
    tables[k] = prep(tables[k]) or tables[k]

# ----------------------------
# STEP 5: build labels
# ----------------------------
labels = None
if "DXSUM" in tables:
    dx = tables["DXSUM"]
    if "visit_id" in dx:
        if any(dx[c].astype(str).str.contains("CN|MCI|AD", na=False, case=False).any() for c in dx.columns):
            for c in dx.columns:
                if dx[c].astype(str).str.contains("CN|MCI|AD", na=False, case=False).any():
                    labels = dx[["visit_id", c]].rename(columns={c:"label"})
                    break
        elif "DXCHANGE" in dx.columns:
            mapping = {1:"CN",2:"MCI",3:"AD",4:"CN",5:"MCI",6:"AD"}
            labels = dx[["visit_id","DXCHANGE"]].copy()
            labels["label"] = labels["DXCHANGE"].map(mapping)
            labels = labels[["visit_id","label"]]

# ----------------------------
# STEP 6: merge features
# ----------------------------
merged = None
for k, df in tables.items():
    if "visit_id" not in df: continue
    # keep numeric-like cols
    keep = ["visit_id"]
    for c in df.columns:
        if c=="visit_id": continue
        if pd.api.types.is_numeric_dtype(df[c]):
            keep.append(c)
        else:
            # try to coerce
            try:
                temp = pd.to_numeric(df[c], errors="coerce")
                if temp.notna().sum()>0.5*len(temp):
                    df[c] = temp
                    keep.append(c)
            except: pass
    block = df[keep].drop_duplicates("visit_id")
    merged = block if merged is None else merged.merge(block, on="visit_id", how="outer")

if labels is not None:
    merged = merged.merge(labels, on="visit_id", how="left")

# ----------------------------
# STEP 7: save merged
# ----------------------------
merged.to_csv(merged_out, index=False)
print("Merged dataset saved:", merged_out, merged.shape)


Loaded DXSUM: (15666, 41) from /mnt/data/extracted_all\DXSUM_06Sep2025.csv
Loaded ADAS: (12774, 16) from /mnt/data/extracted_all\ADAS_06Sep2025.csv
Loaded MMSE: (14533, 58) from /mnt/data/extracted_all\MMSE_06Sep2025.csv
Loaded MOCA: (8871, 58) from /mnt/data/extracted_all\MOCA_06Sep2025.csv
Loaded CDR: (14535, 25) from /mnt/data/extracted_all\CDR_06Sep2025.csv
Loaded FAQ: (35, 18) from /mnt/data/extracted_all\BHR_SP_FAQ_06Sep2025.csv
Loaded ECOGPT: (8093, 62) from /mnt/data/extracted_all\ECOGPT_06Sep2025.csv
Loaded ECOGSP: (8107, 59) from /mnt/data/extracted_all\ECOGSP_06Sep2025.csv
Loaded GDSCALE: (13622, 32) from /mnt/data/extracted_all\GDSCALE_06Sep2025.csv
Loaded NPIQ: (7274, 41) from /mnt/data/extracted_all\NPIQ_06Sep2025.csv
Loaded CBBRESULTS: (22205, 43) from /mnt/data/extracted_all\ADNI_CBBRESULTS_06Sep2025.csv
Loaded PTDEMOG: (6134, 84) from /mnt/data/extracted_all\PTDEMOG_08Jul2025.csv


ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:

import os, re, json, argparse, warnings
from pathlib import Path
import pandas as pd, numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.experimental import enable_hist_gradient_boosting  # noqa: F401
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

# ---------------------- utilities ----------------------

def find_csv(root: Path, patterns):
    root = Path(root)
    hits = {}
    for pat in patterns:
        # accept regex or glob-like
        rx = re.compile(pat, re.I)
        for p in root.rglob("*.csv"):
            if rx.search(p.name):
                hits.setdefault(pat, []).append(p)
    return hits

def read_csv_best(paths):
    # pick the largest file (latest date often has most rows)
    if not paths: 
        return None
    best = sorted(paths, key=lambda p: p.stat().st_size, reverse=True)[0]
    try:
        df = pd.read_csv(best, low_memory=False)
        df.attrs["__source__"] = str(best)
        return df
    except Exception as e:
        print(f"Failed to read {best}: {e}")
        return None

def normalize_cols(df):
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    return df

def col(df, names):
    # return the first matching column (case-insensitive, variants)
    cand = set([c.lower() for c in df.columns])
    for n in names:
        if isinstance(n, str):
            nlow = n.lower()
            if nlow in cand:
                # return original case column
                for c in df.columns:
                    if c.lower()==nlow: return c
        else: # regex
            rx = n
            for c in df.columns:
                if rx.search(c):
                    return c
    return None

def ensure_keys(df):
    # Try to find RID, VISCODE2, VISCODE, VISDATE
    df = normalize_cols(df)
    rid = col(df, ["RID", "rid"])
    if rid is None and "PTID" in df.columns:
        # create RID from PTID trailing digits
        df["RID"] = df["PTID"].astype(str).str.extract(r"(\d+)$").astype(float).astype("Int64")
        rid = "RID"
    elif rid is None and "ptid" in [c.lower() for c in df.columns]:
        ptid = col(df, ["PTID"])
        df["RID"] = df[ptid].astype(str).str.extract(r"(\d+)$").astype(float).astype("Int64")
        rid = "RID"

    viscode2 = col(df, ["VISCODE2", re.compile(r"VISCODE2", re.I)])
    viscode = col(df, ["VISCODE", re.compile(r"VISCODE(?!2)", re.I)])
    visdate = col(df, ["VISDATE", "EXAMDATE", "SCANDATE", re.compile(r"(VIS|EXAM|SCAN)DATE", re.I)])

    # Coerce date
    if visdate:
        df[visdate] = pd.to_datetime(df[visdate], errors="coerce")

    # Build visit_id
    if rid is not None and (viscode2 or viscode):
        vc = viscode2 if viscode2 else viscode
        df["visit_id"] = df[rid].astype(str) + "_" + df[vc].astype(str)
    elif rid is not None and visdate:
        df["visit_id"] = df[rid].astype(str) + "_" + df[visdate].dt.date.astype(str)
    else:
        df["visit_id"] = np.nan

    return df, {"rid": rid, "viscode2": viscode2, "viscode": viscode, "visdate": visdate}

def select_numeric_like(df, include_patterns=None):
    df = df.copy()
    num_cols = []
    for c in df.columns:
        if c in ("visit_id",): 
            continue
        s = df[c]
        if pd.api.types.is_numeric_dtype(s):
            num_cols.append(c)
        else:
            # attempt coercion for common score cols
            try:
                coerced = pd.to_numeric(s, errors="coerce")
                if coerced.notna().sum() > 0.8*len(coerced):
                    df[c] = coerced
                    num_cols.append(c)
            except:
                pass
    if include_patterns:
        rx = re.compile("|".join(include_patterns), re.I)
        num_cols = [c for c in num_cols if rx.search(c)]
    return df, num_cols

def map_dx_from_dxsum(dx):
    dx = normalize_cols(dx)
    cand_cols = [c for c in dx.columns if re.search(r"(DX|DIAG)", c, re.I)]
    # Prefer textual diagnosis
    text_cols = [c for c in cand_cols if dx[c].astype(str).str.contains(r"CN|MCI|AD", case=False, na=False).any()]
    for c in text_cols:
        s = dx[c].astype(str).str.upper()
        if s.isin(["CN","MCI","AD"]).mean() > 0.2:
            m = s.replace({"CN":"CN","MCI":"MCI","AD":"AD"})
            out = dx[["visit_id"]].copy()
            out["label"] = m
            return out

    # Try numeric DXCHANGE mapping (ADNI style; tolerant)
    num_cols = [c for c in cand_cols if pd.api.types.is_numeric_dtype(dx[c])]
    mapping = {
        # Common ADNI DXCHANGE codes (collapsed)
        1:"CN", 2:"MCI", 3:"AD",
        4:"CN", 5:"MCI", 6:"AD",
        7:"CN", 8:"MCI", 9:"AD",
        0:"CN"
    }
    for c in num_cols:
        if dx[c].isin(mapping.keys()).any():
            out = dx[["visit_id"]].copy()
            out["label"] = dx[c].map(mapping).fillna(np.nan)
            return out
    return None

def label_from_cdr(cdr, faq=None):
    cdr = normalize_cols(cdr)
    cdr, k = ensure_keys(cdr)
    cdrg = col(cdr, ["CDRGLOB", re.compile(r"CDR.?GLOB", re.I)])
    cdrsb = col(cdr, ["CDRSUM", "CDR_SB", re.compile(r"CDR.?SUM", re.I)])
    faqtot = None

    if faq is not None:
        faq = normalize_cols(faq)
        faq, _ = ensure_keys(faq)
        faqtot = col(faq, ["FAQTOTAL", re.compile(r"FAQ.*TOT", re.I)])

    df = cdr[["visit_id"]].copy()
    g = cdr[cdrg] if cdrg in cdr.columns else pd.Series(index=cdr.index, dtype=float)
    sb = cdr[cdrsb] if cdrsb in cdr.columns else pd.Series(index=cdr.index, dtype=float)
    if faqtot and "visit_id" in faq.columns:
        fq = faq[["visit_id", faqtot]].drop_duplicates("visit_id")
        df = df.merge(fq, on="visit_id", how="left")
        faqv = df[faqtot]
    else:
        faqv = pd.Series(index=df.index, dtype=float)

    def decide(gi, sbi, fqi):
        try:
            if pd.notna(gi):
                if gi==0: 
                    if pd.isna(sbi) or sbi<1.0: return "CN"
                if gi==0.5: return "MCI"
                if gi>=1: return "AD"
            if pd.notna(sbi):
                if sbi<1.0: return "CN"
                if 1.0<=sbi<4.0 and (pd.isna(fqi) or fqi<10): return "MCI"
                if sbi>=4.0 and (pd.notna(fqi) and fqi>=10): return "AD"
        except:
            return np.nan
        return np.nan

    df["label"] = [decide(gi, sbi, fqi) for gi,sbi,fqi in zip(g, sb, faqv)]
    return df[["visit_id","label"]]

def rbind_keep_unique(a, b):
    if a is None: return b
    if b is None: return a
    return pd.concat([a, b], axis=0).drop_duplicates()

# ---------------------- main pipeline ----------------------

def main(args):
    base_dirs = [Path(p) for p in args.search_dirs]
    print("Searching in:", base_dirs)

    # Discover key tables
    patt = {
        "DXSUM": [r"^DXSUM_.*\.csv$", r"DXSUM"],
        "VISITS": [r"^VISITS_.*\.csv$", r"\bVISITS\b"],
        "PTDEMOG": [r"^PTDEMOG_.*\.csv$", r"\bPTDEMOG\b", r"Subject.?Demograph", r"Demograph"],
        "ADAS": [r"^ADAS_.*\.csv$", r"\bADAS\b"],
        "MMSE": [r"^MMSE_.*\.csv$", r"\bMMSE\b"],
        "MOCA": [r"^MOCA_.*\.csv$", r"\bMOCA\b"],
        "CDR": [r"^CDR_.*\.csv$", r"\bCDR\b"],
        "FAQ": [r"^FAQ_.*\.csv$", r"\bFAQ\b"],
        "ECOGPT": [r"ECOG.*PT_.*\.csv$", r"\bECOGPT\b"],
        "ECOGSP": [r"ECOG.*SP_.*\.csv$", r"\bECOGSP\b"],
        "GDSCALE": [r"^GDSCALE_.*\.csv$", r"\bGDS\b"],
        "NPIQ": [r"^NPIQ_.*\.csv$", r"\bNPIQ\b"],
        "CBBRESULTS": [r"CBBRESULTS_.*\.csv$", r"CBBRESULTS", r"COGSTATE", r"CBB"],
    }

    found = {k: [] for k in patt}
    for d in base_dirs:
        for k, pats in patt.items():
            hits = []
            for pat in pats:
                hits.extend(find_csv(d, [pat]).get(pat, []))
            found[k].extend(hits)

    tables = {k: read_csv_best(v) for k,v in found.items()}

    # ensure keys + visit_id across tables
    keyed = {}
    for k, df in tables.items():
        if df is None: 
            print(f"[WARN] Missing table: {k}")
            continue
        df, meta = ensure_keys(df)
        keyed[k] = df

    # build labels
    labels = None
    if keyed.get("DXSUM") is not None and "visit_id" in keyed["DXSUM"].columns:
        labels = map_dx_from_dxsum(keyed["DXSUM"])
        if labels is not None:
            print(f"Labels from DXSUM: {labels['label'].value_counts(dropna=True).to_dict()}")
    if labels is None and keyed.get("CDR") is not None:
        labels = label_from_cdr(keyed["CDR"], keyed.get("FAQ"))
        print("Labels from CDR heuristic (counts, excl. NaN):", labels["label"].value_counts(dropna=True).to_dict())

    # assemble feature blocks
    blocks = []
    # cognitive totals/items
    for name in ["ADAS","MMSE","MOCA","CDR","FAQ","ECOGPT","ECOGSP","GDSCALE","NPIQ","CBBRESULTS","PTDEMOG"]:
        df = keyed.get(name)
        if df is None: 
            continue
        keep = ["visit_id"]
        # keep numeric-like columns; for demographics keep some categoricals too
        if name=="PTDEMOG":
            # Demographics: keep key fields if present
            for want in ["AGE","PTGENDER","PTEDUCAT","PTETHCAT","PTRACCAT","PTMARRY","PTREGION"]:
                c = [c for c in df.columns if c.upper()==want]
                keep += c
        # numeric expansions
        num_df, num_cols = select_numeric_like(df)
        keep += num_cols
        keep = list(dict.fromkeys([c for c in keep if c in df.columns]))
        blocks.append(df[keep])

    # merge blocks on visit_id
    base = None
    for b in blocks:
        base = b if base is None else base.merge(b, on="visit_id", how="outer")

    if base is None:
        raise SystemExit("No feature tables found.")

    # de-duplicate columns with suffixes
    base = base.loc[:,~base.columns.duplicated()]
    # attach labels
    if labels is not None:
        base = base.merge(labels, on="visit_id", how="left")

    # drop rows without labels
    base_lab = base.dropna(subset=["label"]) if "label" in base.columns else base.copy()

    # Save merged dataset
    out_csv = Path(args.out_dir) / "adni_merged_features.csv"
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    base_lab.to_csv(out_csv, index=False)
    print(f"Saved merged dataset: {out_csv} shape={base_lab.shape}")

    # ---------------- model training (baseline) ----------------
    if args.skip_train:
        print("Skipping model training as requested (--skip-train)."); return
    if "label" not in base_lab.columns:
        print("No labels available. Skipping model training.")
        return

    # Prepare X/y
    y = base_lab["label"].astype(str)
    X = base_lab.drop(columns=["label"])
    # Identify categorical columns (demographics we preserved) and numeric
    cat_cols = [c for c in X.columns if X[c].dtype==object and c!="visit_id" and X[c].nunique(dropna=True)<=30]
    num_cols = [c for c in X.columns if c not in cat_cols and c!="visit_id"]

    # Get RID as groups if present (extract from visit_id prefix when possible)
    if "RID" in base_lab.columns:
        groups = base_lab["RID"].astype(str)
    else:
        groups = X["visit_id"].astype(str).str.split("_").str[0]

    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
    ])

    clf = HistGradientBoostingClassifier(max_depth=8, learning_rate=0.06, max_iter=400,
                                         l2_regularization=0.5, )

    pipe = Pipeline([("pre", pre), ("clf", clf)])

    # 5-fold GroupKFold by subject
    gkf = GroupKFold(n_splits=5)
    y_true, y_predp = [], []
    classes = sorted(y.unique().tolist())
    class_to_idx = {c:i for i,c in enumerate(classes)}

    for fold, (tr, vl) in enumerate(gkf.split(X, y, groups), 1):
        Xtr, Xvl = X.iloc[tr], X.iloc[vl]
        ytr, yvl = y.iloc[tr], y.iloc[vl]
        pipe.fit(Xtr, ytr)
        # Probabilities
        if hasattr(pipe.named_steps["clf"], "predict_proba"):
            P = pipe.predict_proba(Xvl)
            # micro-average AUC via one-vs-rest if possible
            try:
                y_bin = pd.get_dummies(yvl).reindex(columns=classes, fill_value=0).values
                auc = roc_auc_score(y_bin, P, average="macro")
                print(f"[Fold {fold}] Macro AUC: {auc:.3f}")
            except Exception as e:
                print(f"[Fold {fold}] AUC unavailable: {e}")
        yhat = pipe.predict(Xvl)
        print(f"[Fold {fold}] Report:\n", classification_report(yvl, yhat, digits=3))

        y_true.append(yvl)
        y_predp.append(pd.Series(yhat, index=yvl.index))

    y_true = pd.concat(y_true).astype(str)
    y_pred = pd.concat(y_predp).astype(str)
    print("Confusion matrix (overall):\n", confusion_matrix(y_true, y_pred, labels=classes))
    print("Final report:\n", classification_report(y_true, y_pred, digits=3))

    # Save a compact model card
    modelcard = {
        "classes": classes,
        "n_samples": int(len(y_true)),
        "features": {"numeric": num_cols, "categorical": cat_cols},
        "source_dirs": [str(p) for p in args.search_dirs],
        "merged_csv": str(out_csv)
    }
    with open(Path(args.out_dir)/"modelcard.json", "w") as f:
        json.dump(modelcard, f, indent=2)
    print("Saved modelcard.json")

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--search-dirs", nargs="+", required=False, default=[
        "Assessments_v2",
        "Assessments",
        "Enrollment",
        "/mnt/data/extracted/Subject_Characteristics",
        "/mnt/data"
    ])
    ap.add_argument("--out-dir", default="/mnt/data/outputs")
    ap.add_argument("--skip-train", action="store_true")
    args = ap.parse_args()
    main(args)
